# Gesture Recognition
In this group project, you are going to build a 3D Conv model that will be able to predict the 5 gestures correctly. Please import the following libraries to get started.

In [57]:
import numpy as np
import os
import scipy
import imageio
from imageio import imread
import datetime
import os
from keras.layers.recurrent import LSTM
from keras.layers.convolutional import Conv3D, MaxPooling3D, Conv2D, MaxPooling2D
from skimage.io import imread
from skimage.transform import resize
import skimage

We set the random seed so that the results don't vary drastically.

In [58]:
np.random.seed(30)
import random as rn
rn.seed(30)
from keras import backend as K
import tensorflow as tf
tf.random.set_seed(30)

In this block, you read the folder names for training and validation. You also set the `batch_size` here. Note that you set the batch size in such a way that you are able to use the GPU in full capacity. You keep increasing the batch size until the machine throws an error.

In [59]:
train_doc = np.random.permutation(open(r"C:\Users\hkhan\Downloads\Project_data\Project_data\train.csv","r").readlines())
val_doc = np.random.permutation(open(r"C:\Users\hkhan\Downloads\Project_data\Project_data\val.csv").readlines())
batch_size = 10

## Generator
This is one of the most important part of the code. The overall structure of the generator has been given. In the generator, you are going to preprocess the images as you have images of 2 different dimensions as well as create a batch of video frames. You have to experiment with `img_idx`, `y`,`z` and normalization such that you get high accuracy.

In [60]:
x = 30 # number of frames
y = 120 # image width
z = 120 # image height

def generator(source_path, folder_list, batch_size):
    print( 'Source path = ', source_path, '; batch size =', batch_size)
    img_idx = [x for x in range(0,x)] #create a list of image numbers you want to use for a particular video
    while True:
        t = np.random.permutation(folder_list)
        num_batches = len(folder_list)//batch_size # calculate the number of batches
        for batch in range(num_batches): # we iterate over the number of batches
            batch_data = np.zeros((batch_size,x,y,z,3)) # x is the number of images you use for each video, (y,z) is the final size of the input images and 3 is the number of channels RGB
            batch_labels = np.zeros((batch_size,5)) # batch_labels is the one hot representation of the output
            for folder in range(batch_size): # iterate over the batch_size
                imgs = os.listdir(source_path+'/'+ t[folder + (batch*batch_size)].split(';')[0]) # read all the images in the folder
                for idx,item in enumerate(img_idx): #  Iterate iver the frames/images of a folder to read them in
                    image = imageio.imread(source_path+'/'+ t[folder + (batch*batch_size)].strip().split(';')[0]+'/'+imgs[item]).astype(np.float32)
                    
                    #crop the images and resize them. Note that the images are of 2 different shape 
                    #and the conv3D will throw error if the inputs in a batch have different shapes
                    
                    temp = skimage.transform.resize(image,(120,120))
                    temp = temp/127.5-1 #Normalize data
                    
                    batch_data[folder,idx,:,:,0] = (temp[:,:,0]) #normalise and feed in the image
                    batch_data[folder,idx,:,:,1] = (temp[:,:,1]) #normalise and feed in the image
                    batch_data[folder,idx,:,:,2] = (temp[:,:,2]) #normalise and feed in the image
                    
                batch_labels[folder, int(t[folder + (batch*batch_size)].strip().split(';')[2])] = 1
            yield batch_data, batch_labels #you yield the batch_data and the batch_labels, remember what does yield do

        
        # write the code for the remaining data points which are left after full batches
        if (len(folder_list) != batch_size*num_batches):
            print("Batch: ",num_batches+1,"Index:", batch_size)
            batch_size = len(folder_list) - (batch_size*num_batches)
            batch_data = np.zeros((batch_size,x,y,z,3)) # x is the number of images you use for each video, (y,z) is the final size of the input images and 3 is the number of channels RGB
            batch_labels = np.zeros((batch_size,5)) # batch_labels is the one hot representation of the output
            for folder in range(batch_size): # iterate over the batch_size
                imgs = os.listdir(source_path+'/'+ t[folder + (batch*batch_size)].split(';')[0]) # read all the images in the folder
                for idx,item in enumerate(img_idx): #  Iterate iver the frames/images of a folder to read them in
                    image = imageio.imread(source_path+'/'+ t[folder + (batch*batch_size)].strip().split(';')[0]+'/'+imgs[item]).astype(np.float32)
                    
                    #crop the images and resize them. Note that the images are of 2 different shape 
                    #and the conv3D will throw error if the inputs in a batch have different shapes
                    temp = skimage.transform.resize(image,(120,120))
                    temp = temp/127.5-1 #Normalize data
                    
                    batch_data[folder,idx,:,:,0] = (temp[:,:,0])
                    batch_data[folder,idx,:,:,1] = (temp[:,:,1])
                    batch_data[folder,idx,:,:,2] = (temp[:,:,2])
                batch_labels[folder, int(t[folder + (batch*batch_size)].strip().split(';')[2])] = 1
            yield batch_data, batch_labels

Note here that a video is represented above in the generator as (number of images, height, width, number of channels). Take this into consideration while creating the model architecture.

In [61]:
curr_dt_time = datetime.datetime.now()
train_path = "C:/Users/hkhan/Downloads/Project_data/Project_data/train"
val_path = "C:/Users/hkhan/Downloads/Project_data/Project_data/val"
num_train_sequences = len(train_doc)
print('# training sequences =', num_train_sequences)
num_val_sequences = len(val_doc)
print('# validation sequences =', num_val_sequences)
num_epochs = 10
print ('# epochs =', num_epochs)

# training sequences = 663
# validation sequences = 100
# epochs = 10


## Model
Here you make the model using different functionalities that Keras provides. Remember to use `Conv3D` and `MaxPooling3D` and not `Conv2D` and `Maxpooling2D` for a 3D convolution model. You would want to use `TimeDistributed` while building a Conv2D + RNN model. Also remember that the last layer is the softmax. Design the network in such a way that the model is able to give good accuracy on the least number of parameters so that it can fit in the memory of the webcam.

In [62]:
from keras.models import Sequential, Model
from keras.layers import Dense, GRU, Flatten, TimeDistributed, Flatten, BatchNormalization, Activation, Dropout
from keras.layers.convolutional import Conv3D, MaxPooling3D
from keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from keras import optimizers
import keras

#write your model here
#model a
model_a = Sequential()

model_a.add(Conv3D(8, #number of filters 
                 kernel_size=(3,3,3), 
                 input_shape=(30, 120, 120, 3),
                 padding='same'))
model_a.add(BatchNormalization())
model_a.add(Activation('relu'))

model_a.add(MaxPooling3D(pool_size=(2,2,2)))

model_a.add(Conv3D(16, #Number of filters, 
                 kernel_size=(3,3,3), 
                 padding='same'))
model_a.add(BatchNormalization())
model_a.add(Activation('relu'))

model_a.add(MaxPooling3D(pool_size=(2,2,2)))

model_a.add(Conv3D(32, #Number of filters 
                 kernel_size=(1,3,3), 
                 padding='same'))
model_a.add(BatchNormalization())
model_a.add(Activation('relu'))

model_a.add(MaxPooling3D(pool_size=(2,2,2)))

model_a.add(Conv3D(64, #Number pf filters 
                 kernel_size=(1,3,3), 
                 padding='same'))
model_a.add(BatchNormalization())
model_a.add(Activation('relu'))

model_a.add(MaxPooling3D(pool_size=(2,2,2)))

#Flatten Layers
model_a.add(Flatten())

model_a.add(Dense(1000, activation='relu'))
model_a.add(Dropout(0.5))

model_a.add(Dense(500, activation='relu'))
model_a.add(Dropout(0.5))

#softmax layer
model_a.add(Dense(5, activation='softmax'))

Now that you have written the model, the next step is to `compile` the model. When you print the `summary` of the model, you'll see the total number of parameters you have to train.

In [63]:
optimiser = optimizers.Adam(lr=0.001) #write your optimizer
model_a.compile(optimizer=optimiser, loss='categorical_crossentropy', metrics=['categorical_accuracy'])
print (model_a.summary())

Model: "sequential_6"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv3d_24 (Conv3D)           (None, 30, 120, 120, 8)   656       
_________________________________________________________________
batch_normalization_16 (Batc (None, 30, 120, 120, 8)   32        
_________________________________________________________________
activation_24 (Activation)   (None, 30, 120, 120, 8)   0         
_________________________________________________________________
max_pooling3d_20 (MaxPooling (None, 15, 60, 60, 8)     0         
_________________________________________________________________
conv3d_25 (Conv3D)           (None, 15, 60, 60, 16)    3472      
_________________________________________________________________
batch_normalization_17 (Batc (None, 15, 60, 60, 16)    64        
_________________________________________________________________
activation_25 (Activation)   (None, 15, 60, 60, 16)   

Let us create the `train_generator` and the `val_generator` which will be used in `.fit_generator`.

In [64]:
train_generator = generator(train_path, train_doc, batch_size)
val_generator = generator(val_path, val_doc, batch_size)

In [65]:
model_name = 'model_init' + '_' + str(curr_dt_time).replace(' ','').replace(':','_') + '/'
    
if not os.path.exists(model_name):
    os.mkdir(model_name)
        
filepath = model_name + 'model-{epoch:05d}-{loss:.5f}-{categorical_accuracy:.5f}-{val_loss:.5f}-{val_categorical_accuracy:.5f}.h5'

checkpoint = ModelCheckpoint(filepath, monitor='val_loss', verbose=1, save_best_only=False, save_weights_only=False, mode='auto', period=1)

LR = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, cooldown=1, verbose=1) # write the REducelronplateau code here
callbacks_list = [checkpoint, LR]

The `steps_per_epoch` and `validation_steps` are used by `fit_generator` to decide the number of next() calls it need to make.

In [66]:
if (num_train_sequences%batch_size) == 0:
    steps_per_epoch = int(num_train_sequences/batch_size)
else:
    steps_per_epoch = (num_train_sequences//batch_size) + 1

if (num_val_sequences%batch_size) == 0:
    validation_steps = int(num_val_sequences/batch_size)
else:
    validation_steps = (num_val_sequences//batch_size) + 1

Let us now fit the model. This will start training the model and with the help of the checkpoints, you'll be able to save the model at the end of each epoch.

In [67]:
model_a.fit_generator(train_generator, steps_per_epoch=steps_per_epoch, epochs=num_epochs, verbose=1, 
                    callbacks=callbacks_list, validation_data=val_generator, 
                    validation_steps=validation_steps, class_weight=None, workers=1, initial_epoch=0)

Source path =  C:/Users/hkhan/Downloads/Project_data/Project_data/train ; batch size = 10
Epoch 1/10
67/67 [==============================] - 711s 11s/step - loss: 6.7705 - categorical_accuracy: 0.2148 - val_loss: 1.4737 - val_categorical_accuracy: 0.2900

Epoch 00001: saving model to model_init_2021-08-0116_34_42.774504\model-00001-5.14242-0.25189-1.47365-0.29000.h5
Epoch 2/10
67/67 [==============================] - 271s 4s/step - loss: 2.0067 - categorical_accuracy: 0.3687 - val_loss: 1.4439 - val_categorical_accuracy: 0.3900

Epoch 00002: saving model to model_init_2021-08-0116_34_42.774504\model-00002-2.27829-0.29851-1.44385-0.39000.h5
Epoch 3/10
67/67 [==============================] - 274s 4s/step - loss: 1.8298 - categorical_accuracy: 0.3310 - val_loss: 1.3385 - val_categorical_accuracy: 0.3800

Epoch 00003: saving model to model_init_2021-08-0116_34_42.774504\model-00003-1.91598-0.30348-1.33848-0.38000.h5
Epoch 4/10
67/67 [==============================] - 271s 4s/step - loss:

In [68]:
classes = 5 #left swipe, right swipe, thumbs up, thumbs down, stop
channel = 3
x = 30 # number of frames
y = 120 # image width
z = 120 # image height

def generator_ex(source_path, folder_list, batch_size):
    print( 'Source path = ', source_path, '; batch size =', batch_size)
    img_idx = [x for x in range(0,x)] #create a list of image numbers you want to use for a particular video
    while True:
        t = np.random.permutation(folder_list)
        num_batches = len(folder_list)//batch_size # calculate the number of batches
        for batch in range(num_batches): # we iterate over the number of batches
            batch_data = np.zeros((batch_size,x,y,z,channel)) # x is the number of images you use for each video, (y,z) is the final size of the input images and 3 is the number of channels RGB
            batch_labels = np.zeros((batch_size,classes)) # batch_labels is the one hot representation of the output
            for folder in range(batch_size): # iterate over the batch_size
                imgs = os.listdir(source_path+'/'+ t[folder + (batch*batch_size)].split(';')[0]) # read all the images in the folder
                for idx,item in enumerate(img_idx): #  Iterate iver the frames/images of a folder to read them in
                    image = imageio.imread(source_path+'/'+ t[folder + (batch*batch_size)].strip().split(';')[0]+'/'+imgs[item]).astype(np.float32)
                    
                    #crop the images and resize them. Note that the images are of 2 different shape 
                    #and the conv3D will throw error if the inputs in a batch have different shapes
                    
                    temp = skimage.transform.resize(image,(y,z))
                    #Converting to gray scale
                    temp = temp.mean(axis=-1,keepdims=1) 
                    temp = temp/127.5-1 #Normalize data
                    batch_data[folder,idx] = temp #normalise and feed in the image
                    
                batch_labels[folder, int(t[folder + (batch*batch_size)].strip().split(';')[2])] = 1
                
            yield batch_data, batch_labels #you yield the batch_data and the batch_labels, remember what does yield do

        
        # write the code for the remaining data points which are left after full batches
        if (len(folder_list) != batch_size*num_batches):
            print("Batch: ",num_batches+1,"Index:", batch_size)
            batch_size = len(folder_list) - (batch_size*num_batches)
            batch_data = np.zeros((batch_size,x,y,z,channel)) # x is the number of images you use for each video, (y,z) is the final size of the input images and 3 is the number of channels RGB
            batch_labels = np.zeros((batch_size,classes)) # batch_labels is the one hot representation of the output
            for folder in range(batch_size): # iterate over the batch_size
                imgs = os.listdir(source_path+'/'+ t[folder + (batch*batch_size)].split(';')[0]) # read all the images in the folder
                for idx,item in enumerate(img_idx): #  Iterate iver the frames/images of a folder to read them in
                    image = imageio.imread(source_path+'/'+ t[folder + (batch*batch_size)].strip().split(';')[0]+'/'+imgs[item]).astype(np.float32)
                    
                    #crop the images and resize them. Note that the images are of 2 different shape 
                    #and the conv3D will throw error if the inputs in a batch have different shapes
                    temp = skimage.transform.resize(image,(y,z))
                    #Converting to gray scale
                    temp = temp.mean(axis=-1,keepdims=1) 
                    temp = temp/127.5-1 #Normalize data
                    
                    batch_data[folder,idx] = temp
                   
                batch_labels[folder, int(t[folder + (batch*batch_size)].strip().split(';')[2])] = 1
            yield batch_data, batch_labels

In [86]:
from keras.losses import categorical_crossentropy
from keras.optimizers import Adam

# Define model b
model_b = Sequential()
model_b.add(Conv3D(32, kernel_size=(3, 3, 3), input_shape=(x,y,z,channel), padding='same'))
model_b.add(Activation('relu'))
model_b.add(Conv3D(32, kernel_size=(3, 3, 3), padding='same'))
model_b.add(Activation('relu'))
model_b.add(MaxPooling3D(pool_size=(3, 3, 3), padding='same'))
model_b.add(Dropout(0.25))

model_b.add(Conv3D(64, kernel_size=(3, 3, 3), padding='same'))
model_b.add(Activation('relu'))
model_b.add(Conv3D(64, kernel_size=(3, 3, 3), padding='same'))
model_b.add(Activation('relu'))
model_b.add(MaxPooling3D(pool_size=(3, 3, 3), padding='same'))
model_b.add(Dropout(0.25))

model_b.add(Flatten())
model_b.add(Dense(512, activation='relu'))
model_b.add(Dropout(0.5))
model_b.add(Dense(classes, activation='softmax'))

model_b.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['categorical_accuracy'])
model_b.summary()

Model: "sequential_8"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv3d_32 (Conv3D)           (None, 30, 60, 60, 32)    2624      
_________________________________________________________________
activation_32 (Activation)   (None, 30, 60, 60, 32)    0         
_________________________________________________________________
conv3d_33 (Conv3D)           (None, 30, 60, 60, 32)    27680     
_________________________________________________________________
activation_33 (Activation)   (None, 30, 60, 60, 32)    0         
_________________________________________________________________
max_pooling3d_26 (MaxPooling (None, 10, 20, 20, 32)    0         
_________________________________________________________________
dropout_19 (Dropout)         (None, 10, 20, 20, 32)    0         
_________________________________________________________________
conv3d_34 (Conv3D)           (None, 10, 20, 20, 64)   

In [87]:
train_generator = generator_ex(train_path, train_doc, batch_size)
val_generator = generator_ex(val_path, val_doc, batch_size)

In [88]:
if (num_train_sequences%batch_size) == 0:
    steps_per_epoch = int(num_train_sequences/batch_size)
else:
    steps_per_epoch = (num_train_sequences//batch_size) + 1

if (num_val_sequences%batch_size) == 0:
    validation_steps = int(num_val_sequences/batch_size)
else:
    validation_steps = (num_val_sequences//batch_size) + 1

In [89]:
model_b.fit_generator(train_generator, steps_per_epoch=steps_per_epoch, epochs=num_epochs, verbose=1, 
                    callbacks=callbacks_list, validation_data=val_generator, 
                    validation_steps=validation_steps, class_weight=None, workers=1, initial_epoch=0)

Source path =  C:/Users/hkhan/Downloads/Project_data/Project_data/train ; batch size = 10
Epoch 1/10
67/67 [==============================] - 906s 14s/step - loss: 1.7399 - categorical_accuracy: 0.2151 - val_loss: 1.5538 - val_categorical_accuracy: 0.3400

Epoch 00001: saving model to model_init_2021-08-0116_34_42.774504\model-00001-1.64142-0.19457-1.55384-0.34000.h5
Epoch 2/10
67/67 [==============================] - 326s 5s/step - loss: 1.6024 - categorical_accuracy: 0.2292 - val_loss: 1.6103 - val_categorical_accuracy: 0.1700

Epoch 00002: saving model to model_init_2021-08-0116_34_42.774504\model-00002-1.61157-0.21393-1.61028-0.17000.h5
Epoch 3/10
67/67 [==============================] - 333s 5s/step - loss: 1.6101 - categorical_accuracy: 0.1495 - val_loss: 1.6097 - val_categorical_accuracy: 0.2000

Epoch 00003: saving model to model_init_2021-08-0116_34_42.774504\model-00003-1.61026-0.17910-1.60971-0.20000.h5

Epoch 00003: ReduceLROnPlateau reducing learning rate to 0.000500000023

### Changing x,y,z values Experiment [ 1]


In [90]:
x = 30 # number of frames
y = 60 # image width
z = 60 # image height

In [91]:
# Define model b
model_b = Sequential()
model_b.add(Conv3D(32, kernel_size=(3, 3, 3), input_shape=(x,y,z,channel), padding='same'))
model_b.add(Activation('relu'))
model_b.add(Conv3D(32, kernel_size=(3, 3, 3), padding='same'))
model_b.add(Activation('relu'))
model_b.add(MaxPooling3D(pool_size=(3, 3, 3), padding='same'))
model_b.add(Dropout(0.25))

model_b.add(Conv3D(64, kernel_size=(3, 3, 3), padding='same'))
model_b.add(Activation('relu'))
model_b.add(Conv3D(64, kernel_size=(3, 3, 3), padding='same'))
model_b.add(Activation('relu'))
model_b.add(MaxPooling3D(pool_size=(3, 3, 3), padding='same'))
model_b.add(Dropout(0.25))

model_b.add(Flatten())
model_b.add(Dense(512, activation='relu'))
model_b.add(Dropout(0.5))
model_b.add(Dense(classes, activation='softmax'))

model_b.compile(optimizer=keras.optimizers.Adam(), loss='categorical_crossentropy', metrics=['categorical_accuracy'])
model_b.summary()

Model: "sequential_9"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv3d_36 (Conv3D)           (None, 30, 60, 60, 32)    2624      
_________________________________________________________________
activation_36 (Activation)   (None, 30, 60, 60, 32)    0         
_________________________________________________________________
conv3d_37 (Conv3D)           (None, 30, 60, 60, 32)    27680     
_________________________________________________________________
activation_37 (Activation)   (None, 30, 60, 60, 32)    0         
_________________________________________________________________
max_pooling3d_28 (MaxPooling (None, 10, 20, 20, 32)    0         
_________________________________________________________________
dropout_22 (Dropout)         (None, 10, 20, 20, 32)    0         
_________________________________________________________________
conv3d_38 (Conv3D)           (None, 10, 20, 20, 64)   

In [92]:
train_generator = generator_ex(train_path, train_doc, batch_size)
val_generator = generator_ex(val_path, val_doc, batch_size)

In [93]:
if (num_train_sequences%batch_size) == 0:
    steps_per_epoch = int(num_train_sequences/batch_size)
else:
    steps_per_epoch = (num_train_sequences//batch_size) + 1

if (num_val_sequences%batch_size) == 0:
    validation_steps = int(num_val_sequences/batch_size)
else:
    validation_steps = (num_val_sequences//batch_size) + 1

In [94]:
model_b.fit_generator(train_generator, steps_per_epoch=steps_per_epoch, epochs=num_epochs, verbose=1, 
                    callbacks=callbacks_list, validation_data=val_generator, 
                    validation_steps=validation_steps, class_weight=None, workers=1, initial_epoch=0)

Source path =  C:/Users/hkhan/Downloads/Project_data/Project_data/train ; batch size = 10
Epoch 1/10
67/67 [==============================] - 836s 12s/step - loss: 1.6495 - categorical_accuracy: 0.2085 - val_loss: 1.5576 - val_categorical_accuracy: 0.3200

Epoch 00001: saving model to model_init_2021-08-0116_34_42.774504\model-00001-1.61502-0.21267-1.55764-0.32000.h5
Epoch 2/10
67/67 [==============================] - 316s 5s/step - loss: 1.5246 - categorical_accuracy: 0.3480 - val_loss: 1.6121 - val_categorical_accuracy: 0.2200

Epoch 00002: saving model to model_init_2021-08-0116_34_42.774504\model-00002-1.56675-0.30846-1.61211-0.22000.h5
Epoch 3/10
67/67 [==============================] - 308s 5s/step - loss: 1.6110 - categorical_accuracy: 0.2099 - val_loss: 1.6122 - val_categorical_accuracy: 0.2000

Epoch 00003: saving model to model_init_2021-08-0116_34_42.774504\model-00003-1.61372-0.18905-1.61223-0.20000.h5

Epoch 00003: ReduceLROnPlateau reducing learning rate to 0.000500000023

### Changing Batch size  to 20 Experiment [ 2]


In [95]:
train_generator = generator_ex(train_path, train_doc, 20)
val_generator = generator_ex(val_path, val_doc, 20)

In [96]:
if (num_train_sequences%batch_size) == 0:
    steps_per_epoch = int(num_train_sequences/batch_size)
else:
    steps_per_epoch = (num_train_sequences//batch_size) + 1

if (num_val_sequences%batch_size) == 0:
    validation_steps = int(num_val_sequences/batch_size)
else:
    validation_steps = (num_val_sequences//batch_size) + 1

In [97]:
model_b.fit_generator(train_generator, steps_per_epoch=steps_per_epoch, epochs=num_epochs, verbose=1, 
                    callbacks=callbacks_list, validation_data=val_generator, 
                    validation_steps=validation_steps, class_weight=None, workers=1, initial_epoch=0)

Source path =  C:/Users/hkhan/Downloads/Project_data/Project_data/train ; batch size = 20
Epoch 1/10
67/67 [==============================] - 1029s 15s/step - loss: 1.3786 - categorical_accuracy: 0.4331 - val_loss: 1.3895 - val_categorical_accuracy: 0.4600

Epoch 00001: saving model to model_init_2021-08-0116_34_42.774504\model-00001-1.37858-0.43307-1.38946-0.46000.h5
Epoch 2/10
67/67 [==============================] - 354s 5s/step - loss: 1.4517 - categorical_accuracy: 0.3532 - val_loss: 1.3878 - val_categorical_accuracy: 0.4850

Epoch 00002: saving model to model_init_2021-08-0116_34_42.774504\model-00002-1.45170-0.35323-1.38778-0.48500.h5
Epoch 3/10
67/67 [==============================] - 354s 5s/step - loss: 1.3547 - categorical_accuracy: 0.4478 - val_loss: 1.3319 - val_categorical_accuracy: 0.4800

Epoch 00003: saving model to model_init_2021-08-0116_34_42.774504\model-00003-1.35466-0.44776-1.33190-0.48000.h5
Epoch 4/10
67/67 [==============================] - 358s 5s/step - loss

### Changing Batch size  to 30 Experiment [ 3]


In [98]:
train_generator = generator_ex(train_path, train_doc, 30)
val_generator = generator_ex(val_path, val_doc, 30)

In [99]:
if (num_train_sequences%batch_size) == 0:
    steps_per_epoch = int(num_train_sequences/batch_size)
else:
    steps_per_epoch = (num_train_sequences//batch_size) + 1

if (num_val_sequences%batch_size) == 0:
    validation_steps = int(num_val_sequences/batch_size)
else:
    validation_steps = (num_val_sequences//batch_size) + 1

In [100]:
model_b.fit_generator(train_generator, steps_per_epoch=steps_per_epoch, epochs=num_epochs, verbose=1, 
                    callbacks=callbacks_list, validation_data=val_generator, 
                    validation_steps=validation_steps, class_weight=None, workers=1, initial_epoch=0)

Source path =  C:/Users/hkhan/Downloads/Project_data/Project_data/train ; batch size = 30
Epoch 1/10
67/67 [==============================] - ETA: 0s - loss: 0.9283 - categorical_accuracy: 0.6302 Source path =  C:/Users/hkhan/Downloads/Project_data/Project_data/val ; batch size = 30
Batch:  4 Index: 30
67/67 [==============================] - 1184s 17s/step - loss: 0.9283 - categorical_accuracy: 0.6302 - val_loss: 1.0532 - val_categorical_accuracy: 0.6500

Epoch 00001: saving model to model_init_2021-08-0116_34_42.774504\model-00001-0.92831-0.63019-1.05317-0.65000.h5
Epoch 2/10
67/67 [==============================] - 341s 5s/step - loss: 1.0137 - categorical_accuracy: 0.6070 - val_loss: 1.0624 - val_categorical_accuracy: 0.6200

Epoch 00002: saving model to model_init_2021-08-0116_34_42.774504\model-00002-1.01369-0.60697-1.06237-0.62000.h5
Epoch 3/10
67/67 [==============================] - 337s 5s/step - loss: 0.8937 - categorical_accuracy: 0.6468 - val_loss: 1.0665 - val_categorical

### Changing Batch size  to 40 Experiment [ 4]


In [101]:
train_generator = generator_ex(train_path, train_doc, 40)
val_generator = generator_ex(val_path, val_doc, 40)

In [102]:
if (num_train_sequences%batch_size) == 0:
    steps_per_epoch = int(num_train_sequences/batch_size)
else:
    steps_per_epoch = (num_train_sequences//batch_size) + 1

if (num_val_sequences%batch_size) == 0:
    validation_steps = int(num_val_sequences/batch_size)
else:
    validation_steps = (num_val_sequences//batch_size) + 1

In [103]:
model_b.fit_generator(train_generator, steps_per_epoch=steps_per_epoch, epochs=num_epochs, verbose=1, 
                    callbacks=callbacks_list, validation_data=val_generator, 
                    validation_steps=validation_steps, class_weight=None, workers=1, initial_epoch=0)

Source path =  C:/Users/hkhan/Downloads/Project_data/Project_data/train ; batch size = 40
Epoch 1/10
67/67 [==============================] - ETA: 0s - loss: 0.6284 - categorical_accuracy: 0.7646 Source path =  C:/Users/hkhan/Downloads/Project_data/Project_data/val ; batch size = 40
Batch:  3 Index: 40
67/67 [==============================] - 2234s 33s/step - loss: 0.6284 - categorical_accuracy: 0.7646 - val_loss: 0.9783 - val_categorical_accuracy: 0.6583

Epoch 00001: saving model to model_init_2021-08-0116_34_42.774504\model-00001-0.62843-0.76464-0.97830-0.65833.h5
Epoch 2/10
67/67 [==============================] - 1507s 22s/step - loss: 0.5732 - categorical_accuracy: 0.7742 - val_loss: 0.9649 - val_categorical_accuracy: 0.6950

Epoch 00002: saving model to model_init_2021-08-0116_34_42.774504\model-00002-0.57318-0.77425-0.96492-0.69500.h5
Epoch 3/10
67/67 [==============================] - 1538s 23s/step - loss: 0.5873 - categorical_accuracy: 0.7779 - val_loss: 0.9783 - val_categor

### Change Optimizer to Adadelta Experiment [5]

In [104]:
model_b.compile(optimizer=keras.optimizers.Adadelta(), loss='categorical_crossentropy', metrics=['categorical_accuracy'])
model_b.summary()

Model: "sequential_9"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv3d_36 (Conv3D)           (None, 30, 60, 60, 32)    2624      
_________________________________________________________________
activation_36 (Activation)   (None, 30, 60, 60, 32)    0         
_________________________________________________________________
conv3d_37 (Conv3D)           (None, 30, 60, 60, 32)    27680     
_________________________________________________________________
activation_37 (Activation)   (None, 30, 60, 60, 32)    0         
_________________________________________________________________
max_pooling3d_28 (MaxPooling (None, 10, 20, 20, 32)    0         
_________________________________________________________________
dropout_22 (Dropout)         (None, 10, 20, 20, 32)    0         
_________________________________________________________________
conv3d_38 (Conv3D)           (None, 10, 20, 20, 64)   

In [105]:
train_generator = generator_ex(train_path, train_doc, 40)
val_generator = generator_ex(val_path, val_doc, 40)

In [106]:
if (num_train_sequences%batch_size) == 0:
    steps_per_epoch = int(num_train_sequences/batch_size)
else:
    steps_per_epoch = (num_train_sequences//batch_size) + 1

if (num_val_sequences%batch_size) == 0:
    validation_steps = int(num_val_sequences/batch_size)
else:
    validation_steps = (num_val_sequences//batch_size) + 1

In [107]:
model_b.fit_generator(train_generator, steps_per_epoch=steps_per_epoch, epochs=num_epochs, verbose=1, 
                    callbacks=callbacks_list, validation_data=val_generator, 
                    validation_steps=validation_steps, class_weight=None, workers=1, initial_epoch=0)

Source path =  C:/Users/hkhan/Downloads/Project_data/Project_data/train ; batch size = 40
Epoch 1/10
67/67 [==============================] - ETA: 0s - loss: 0.4112 - categorical_accuracy: 0.8389 Source path =  C:/Users/hkhan/Downloads/Project_data/Project_data/val ; batch size = 40
Batch:  3 Index: 40
67/67 [==============================] - 2442s 36s/step - loss: 0.4110 - categorical_accuracy: 0.8390 - val_loss: 0.8938 - val_categorical_accuracy: 0.6917

Epoch 00001: saving model to model_init_2021-08-0116_34_42.774504\model-00001-0.39812-0.84754-0.89375-0.69167.h5
Epoch 2/10
67/67 [==============================] - 1812s 27s/step - loss: 0.4257 - categorical_accuracy: 0.8412 - val_loss: 0.9429 - val_categorical_accuracy: 0.6550

Epoch 00002: saving model to model_init_2021-08-0116_34_42.774504\model-00002-0.40931-0.84292-0.94291-0.65500.h5
Epoch 3/10
67/67 [==============================] - 1556s 23s/step - loss: 0.4182 - categorical_accuracy: 0.8509 - val_loss: 0.8931 - val_categor

###### Till this point we achieved 86.04% accuracy in training data

### Model C Experiment [6]

In [129]:
input_shape=(x,y,z,channel)

nb_filters = [8,16,32,64]
nb_dense = [1000, 500, 5]
# Define model
model_c = Sequential()

model_c.add(Conv3D(nb_filters[0], 
                 kernel_size=(3,3,3), 
                 input_shape=input_shape,
                 padding='same'))
model_c.add(BatchNormalization())
model_c.add(Activation('relu'))

model_c.add(MaxPooling3D(pool_size=(2,2,2)))

model_c.add(Conv3D(nb_filters[1], 
                 kernel_size=(3,3,3), 
                 padding='same'))
model_c.add(BatchNormalization())
model_c.add(Activation('relu'))

model_c.add(MaxPooling3D(pool_size=(2,2,2)))

model_c.add(Conv3D(nb_filters[2], 
                 kernel_size=(1,3,3), 
                 padding='same'))
model_c.add(BatchNormalization())
model_c.add(Activation('relu'))

model_c.add(MaxPooling3D(pool_size=(2,2,2)))

model_c.add(Conv3D(nb_filters[3], 
                 kernel_size=(1,3,3), 
                 padding='same'))
model_c.add(BatchNormalization())
model_c.add(Activation('relu'))

model_c.add(MaxPooling3D(pool_size=(2,2,2)))

#Flatten Layers
model_c.add(Flatten())

model_c.add(Dense(nb_dense[0], activation='relu'))
model_d.add(Dropout(0.5))

model_c.add(Dense(nb_dense[1], activation='relu'))
model_c.add(Dropout(0.5))

#softmax layer
model_c.add(Dense(nb_dense[2], activation='softmax'))
model_c.compile(optimizer=keras.optimizers.Adadelta(), loss='categorical_crossentropy', metrics=['categorical_accuracy'])
model_c.summary()

Model: "sequential_14"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv3d_62 (Conv3D)           (None, 30, 120, 120, 8)   656       
_________________________________________________________________
batch_normalization_27 (Batc (None, 30, 120, 120, 8)   32        
_________________________________________________________________
activation_62 (Activation)   (None, 30, 120, 120, 8)   0         
_________________________________________________________________
max_pooling3d_43 (MaxPooling (None, 15, 60, 60, 8)     0         
_________________________________________________________________
conv3d_63 (Conv3D)           (None, 15, 60, 60, 16)    3472      
_________________________________________________________________
batch_normalization_28 (Batc (None, 15, 60, 60, 16)    64        
_________________________________________________________________
activation_63 (Activation)   (None, 15, 60, 60, 16)  

In [130]:
if (num_train_sequences%batch_size) == 0:
    steps_per_epoch = int(num_train_sequences/batch_size)
else:
    steps_per_epoch = (num_train_sequences//batch_size) + 1

if (num_val_sequences%batch_size) == 0:
    validation_steps = int(num_val_sequences/batch_size)
else:
    validation_steps = (num_val_sequences//batch_size) + 1

In [131]:
train_generator = generator_ex(train_path, train_doc, batch_size)
val_generator = generator_ex(val_path, val_doc, batch_size)
num_epochs = 10
model_c.fit_generator(train_generator, steps_per_epoch=steps_per_epoch, epochs=num_epochs, verbose=1, 
                    callbacks=callbacks_list, validation_data=val_generator, 
                    validation_steps=validation_steps, class_weight=None, workers=1, initial_epoch=0)

Source path =  C:/Users/hkhan/Downloads/Project_data/Project_data/train ; batch size = 10
Epoch 1/10
67/67 [==============================] - 725s 11s/step - loss: 3.3132 - categorical_accuracy: 0.2249 - val_loss: 1.6247 - val_categorical_accuracy: 0.1700

Epoch 00001: saving model to model_init_2021-08-0116_34_42.774504\model-00001-3.25117-0.21719-1.62469-0.17000.h5
Epoch 2/10
67/67 [==============================] - 286s 4s/step - loss: 2.6720 - categorical_accuracy: 0.2667 - val_loss: 1.6480 - val_categorical_accuracy: 0.2400

Epoch 00002: saving model to model_init_2021-08-0116_34_42.774504\model-00002-2.91681-0.24378-1.64800-0.24000.h5
Epoch 3/10
67/67 [==============================] - 277s 4s/step - loss: 3.0268 - categorical_accuracy: 0.1909 - val_loss: 1.6267 - val_categorical_accuracy: 0.2100

Epoch 00003: saving model to model_init_2021-08-0116_34_42.774504\model-00003-3.03876-0.22388-1.62670-0.21000.h5

Epoch 00003: ReduceLROnPlateau reducing learning rate to 0.000500000023

# Experiment 5 achived the best Accuracy